# Colab runner

Launcher only — no method code lives here. See `model.py`.

**Connect first:** `Select Kernel` → `Colab` → `New Colab Server` → pick **GPU**.
Then run these cells top to bottom.

Everything below executes on the Colab VM, not on your laptop.


## 1. Confirm we actually got a GPU

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

## 2. Mount Drive

(Or use the command palette: `Colab: Mount Google Drive to Server...`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo onto the VM

`/content` is wiped when the runtime dies, so this runs every fresh session.

In [ ]:
%cd /content
!git clone https://github.com/Dev-Joyson/CNN-based-GAN-face-detection.git 2>/dev/null || (cd CNN-based-GAN-face-detection && git pull)
%cd /content/CNN-based-GAN-face-detection
!pip install -q pyyaml

## 4. Sanity check: do the guard tests pass on this machine?

~3 seconds, no dataset needed.

In [ ]:
!pytest tests -q

## 5. Audit the dataset for shortcuts

No GPU needed. Scores each trivial file property as an AUC — if any reaches ~0.9,
the model can hit that score without looking at a face.

In [ ]:
!python audit_dataset.py --config configs/test13_face.yaml

## 6. Train the headline model (full image, no mask)

Epoch 1 is slow — it reads every image off Drive and writes the cache to
local disk. Later epochs read the cache and are much faster. Don't kill it.

Outputs go to Drive (`out_dir` in the config), so they survive a disconnect.


In [ ]:
!python train.py --config configs/test16_full.yaml


## 7. Controls: face-only and background-only

The panel's question was whether the model reads the background. These two
runs answer it. They use a different dataset/size to test16, so they build
their own cache.


In [ ]:
!python train.py --config configs/test13_face.yaml
!python train.py --config configs/test14_background.yaml


## 8. Evaluate: confusion matrix, ROC, Grad-CAM

Writes `confusion_matrix.png`, `roc.png`, `gradcam.png` and `eval.json` into the
run folder in Drive. Uses the same masked pipeline as training, so the numbers
match what the model actually saw.

Grad-CAM is the shortcut check: under `face_only` the heat should sit on the face.


In [ ]:
!python evaluate.py --config configs/test16_full.yaml
!python evaluate.py --config configs/test13_face.yaml
!python evaluate.py --config configs/test14_background.yaml


## 9. Watch training

**Live:** TensorBoard reads `<run>/tb/` and refreshes itself every 30 s.
Pointing it at the experiments root shows every run on one chart.
If the panel does not render inside VS Code, open this same notebook at
colab.research.google.com — it embeds there.

**Snapshot:** the cell after reads `history.csv` from Drive; re-run it any time.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/Research/experiments"


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv("/content/drive/MyDrive/Research/experiments/test16_full/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h[["accuracy", "val_accuracy"]].plot(ax=ax[0], title="accuracy")
h[["auc", "val_auc"]].plot(ax=ax[1], title="AUC")
plt.tight_layout(); plt.show()